# Eagle5 Headbank — Corrected Retrain

Retrains the spec-decode head for every Qwen model with the two fixes validated locally on Qwen-3B (lens ceiling 0%→74%, head depth-1 73.9%):

1. **Frozen weights from the GGUF dequant** (not HF fp16) so the head's baseline `RMSNorm(residual)@lm_head` matches the runtime verifier.
2. **`--target-mode corpus`** — train + eval against the model's REAL next token, not the self-referential baseline proxy.
3. Residuals captured from the **quantized (Q4_K_M)** runtime at layer n-1.

**Just `Runtime → Run all`.** Inputs (corpora + frozen weights) auto-download from the GitHub release — nothing to upload. Trained heads are written to your Drive.


## 1. Mount Drive (for output) + clone repo + deps

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT_ROOT = '/content/drive/MyDrive/dismantle_headbank_corrected/heads_corrected'
import os
os.makedirs(OUT_ROOT, exist_ok=True)

REPO_URL = 'https://github.com/joshuahickscorp/dismantle.git'
BRANCH = 'codex/maximal-spec-colab'
if not os.path.isdir('/content/dismantle'):
    !git clone --depth 1 --branch $BRANCH $REPO_URL /content/dismantle
else:
    !cd /content/dismantle && git fetch --depth 1 origin $BRANCH && git checkout $BRANCH && git reset --hard origin/$BRANCH
!pip -q install pyarrow safetensors gguf packaging
import sys
sys.path.insert(0, '/content/dismantle/colab')
print('output ->', OUT_ROOT)
print('repo + deps ready')


## 2. Auto-fetch inputs (corpus from release, frozen rebuilt from HF)
Corpora (locally-captured quantized residuals) download from the GitHub release. Frozen weights are rebuilt on Colab from the **official Qwen HF GGUFs** (fast Colab pipe, not your slow upstream) and **verified against an `output_norm` fingerprint** — a mismatch fails safe (skips the model) rather than training a bad head.

In [ ]:
REPO_SLUG = 'joshuahickscorp/dismantle'
RELEASE_TAG = 'headbank-corpus-v1'
MODELS = {
    "q05b": {
        "capture_layer": 23,
        "label": "Qwen2.5-0.5B-Instruct",
        "hf_repo": "Qwen/Qwen2.5-0.5B-Instruct-GGUF",
        "hf_file": "qwen2.5-0.5b-instruct-q4_k_m.gguf"
    },
    "q1p5b": {
        "capture_layer": 27,
        "label": "Qwen2.5-1.5B-Instruct",
        "hf_repo": "Qwen/Qwen2.5-1.5B-Instruct-GGUF",
        "hf_file": "qwen2.5-1.5b-instruct-q4_k_m.gguf"
    },
    "q3b": {
        "capture_layer": 35,
        "label": "Qwen2.5-3B-Instruct",
        "hf_repo": "Qwen/Qwen2.5-3B-Instruct-GGUF",
        "hf_file": "qwen2.5-3b-instruct-q4_k_m.gguf"
    },
    "q7b": {
        "capture_layer": 27,
        "label": "Qwen2.5-7B-Instruct",
        "hf_repo": "bartowski/Qwen2.5-7B-Instruct-GGUF",
        "hf_file": "Qwen2.5-7B-Instruct-Q4_K_M.gguf"
    }
}
import os, json, tarfile, urllib.request, subprocess, numpy as np
!pip -q install huggingface_hub >/dev/null
from huggingface_hub import hf_hub_download
DATA_ROOT = '/content/headbank_inputs'
BASE = f'https://github.com/{REPO_SLUG}/releases/download/{RELEASE_TAG}'
BUILDER = '/content/dismantle/tools/orchestrator/build_frozen_gguf.py'
os.makedirs(DATA_ROOT, exist_ok=True)
# fingerprints for frozen verification
fp_path = os.path.join(DATA_ROOT, 'frozen_fingerprints.json')
urllib.request.urlretrieve(f'{BASE}/frozen_fingerprints.json', fp_path)
FP = json.load(open(fp_path))

READY = []
for slug, cfg in MODELS.items():
    dst = os.path.join(DATA_ROOT, slug)
    shards = os.path.join(dst, 'corpus_shards')
    os.makedirs(shards, exist_ok=True)
    frozen = os.path.join(dst, 'frozen_gguf.npz')
    # 1) corpus from release
    if not any(f.endswith('.parquet') for f in os.listdir(shards)):
        tarp = os.path.join(dst, 'corpus.tar')
        print(f'{slug}: downloading corpus...')
        urllib.request.urlretrieve(f'{BASE}/{slug}_corpus.tar', tarp)
        with tarfile.open(tarp) as t: t.extractall(shards)
        os.remove(tarp)
    # 2) frozen: rebuild from HF GGUF, verify against fingerprint
    if not os.path.isfile(frozen):
        print(f'{slug}: downloading GGUF {cfg["hf_repo"]}/{cfg["hf_file"]} ...')
        gguf = hf_hub_download(repo_id=cfg['hf_repo'], filename=cfg['hf_file'])
        print(f'{slug}: building frozen from GGUF dequant...')
        r = subprocess.run(['python', BUILDER, '--gguf', gguf, '--out', frozen],
                           capture_output=True, text=True)
        if r.returncode != 0:
            print(f'{slug}: build_frozen failed:', r.stderr[-400:]); continue
    # 3) verify output_norm fingerprint
    z = np.load(frozen)
    on = z['output_norm'].astype(np.float32)
    ref = np.array(FP[slug]['output_norm'], dtype=np.float32)
    ok = on.shape == ref.shape and float(np.abs(on - ref).max()) < 1e-3
    n = len([f for f in os.listdir(shards) if f.endswith('.parquet')])
    print(f'{slug}: {n} shards, frozen_verified={ok}')
    if ok and n > 0:
        READY.append(slug)
    else:
        print(f'   !! {slug} SKIPPED — frozen fingerprint mismatch (GGUF source differs). '
              f'Upload {slug}_frozen.npz to the release to train it.')
print('\nREADY:', READY)


## 3. Training config

In [ ]:
TRAIN = {"epochs": 10, "batch_size": 24, "seq_len": 16, "lr": 0.001, "rollout_loss_weight": 1.0, "rollout_depth": 5, "rollout_depth_targets": "1,2,3,4", "rollout_draft_prob": 0.75, "rollout_chain_hidden": true}
TRAINABLE = READY  # set in cell 2 (corpus present + frozen fingerprint verified)
print('trainable:', TRAINABLE)
assert TRAINABLE, 'no models ready — check cell 2 output'


## 4. Train every ready head (corrected pipeline)

In [ ]:
import subprocess, os
TRAINER = '/content/dismantle/colab/eagle5_train_pytorch.py'

for slug in TRAINABLE:
    cfg = MODELS[slug]
    shards = os.path.join(DATA_ROOT, slug, 'corpus_shards')
    frozen = os.path.join(DATA_ROOT, slug, 'frozen_gguf.npz')
    ckpt = os.path.join(OUT_ROOT, slug)
    os.makedirs(ckpt, exist_ok=True)
    cmd = ['python', TRAINER,
           '--corpus-dir', shards,
           '--frozen', frozen,
           '--ckpt-dir', ckpt,
           '--device', 'cuda',
           '--target-mode', 'corpus',
           '--capture-layer', str(cfg['capture_layer']),
           '--epochs', str(TRAIN['epochs']),
           '--batch-size', str(TRAIN['batch_size']),
           '--seq-len', str(TRAIN['seq_len']),
           '--lr', str(TRAIN['lr']),
           '--rollout-loss-weight', str(TRAIN['rollout_loss_weight']),
           '--rollout-depth', str(TRAIN['rollout_depth']),
           '--rollout-depth-targets', TRAIN['rollout_depth_targets'],
           '--rollout-draft-prob', str(TRAIN['rollout_draft_prob']),
           '--save-safetensors']
    if TRAIN.get('rollout_chain_hidden'):
        cmd.append('--rollout-chain-hidden')
    print('\n===', slug, cfg['label'], '===')
    print(' '.join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-2000:])
    if r.returncode != 0:
        print('STDERR:', r.stderr[-2000:])
    else:
        print('OK ->', os.path.join(ckpt, 'head_final.safetensors'))


## 5. Tau eval (corpus mode = real next token)
Reports the honest depth-1 / multi-depth acceptance ceiling against the model's actual next token (not the old self-referential proxy).

In [ ]:
import subprocess, os, json
EVAL = '/content/dismantle/colab/eagle5_tau_eval_pytorch.py'
tau_results = {}
for slug in TRAINABLE:
    cfg = MODELS[slug]
    ckpt = os.path.join(OUT_ROOT, slug, 'latest.npz')
    frozen = os.path.join(DATA_ROOT, slug, 'frozen_gguf.npz')
    shards = os.path.join(DATA_ROOT, slug, 'corpus_shards')
    out = os.path.join(OUT_ROOT, slug, 'tau_eval.json')
    if not os.path.isfile(ckpt):
        print(slug, 'no checkpoint, skipping'); continue
    cmd = ['python', EVAL,
           '--ckpt', ckpt, '--frozen', frozen, '--corpus', shards,
           '--out', out, '--device', 'cuda',
           '--target-mode', 'corpus', '--depth', '4']
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode == 0 and os.path.isfile(out):
        tau_results[slug] = json.load(open(out))
        d1 = tau_results[slug].get('depth1_accept_rate', 0)
        tau = tau_results[slug].get('tau', 0)
        print(f'{slug:6s} depth1={d1:.1%} tau={tau:.2f}')
    else:
        print(slug, 'eval failed:', r.stderr[-800:])


## 6. Emit headbank manifest + summary

In [ ]:
import json, os, hashlib
def sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1<<20), b''): h.update(b)
    return h.hexdigest()
entries = []
for slug in TRAINABLE:
    head = os.path.join(OUT_ROOT, slug, 'head_final.safetensors')
    if not os.path.isfile(head): continue
    entries.append({
        'slug': slug,
        'label': MODELS[slug]['label'],
        'arch': 'qwen2',
        'capture_layer': MODELS[slug]['capture_layer'],
        'head_path': head,
        'head_sha256': sha256(head),
        'metrics': tau_results.get(slug, {}),
    })
manifest = {'schema': 'dismantle-headbank-corrected-v1', 'entries': entries}
mpath = os.path.join(OUT_ROOT, 'headbank_manifest.json')
json.dump(manifest, open(mpath, 'w'), indent=2)
print('wrote', mpath)
for e in entries:
    m = e['metrics']
    print(f"{e['slug']:6s} depth1={m.get('depth1_accept_rate',0):.1%} tau={m.get('tau',0):.2f}")
print('\nDownload heads_corrected/ from Drive and stage with tools/headbank/pull.py')
